# Husky USERIO Clocks

You can now generate clocks of your choosing and output them on USERIO pins!

In [ ]:
SCOPE="OPENADC"
PLATFORM="CWHUSKY"

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

# Number of clocks

The USERIO clocks are generated by a PLL inside the FPGA.

`scope.userio.num_clocks` tells you how many clocks you can generate: 1 for Husky, 4 for Husky Plus (thanks to its much larger FPGA).

Clocks are assigned to the `USERIO CK` pin first, then `USERIO D7` downwards. Clocks can be output on:
- Husky: `USERIO CK`
- Husky Plus: `USERIO CK, D7, D6, D5`

You can also see this when you print the `scope.userio` object (notice the "clocks not supported" / "clock not set" label).

***(Keep reading to the end of this notebook to find out about an additional, bonus clock!)***

In [ ]:
print('Your Husky supports %d USERIO clocks.' % scope.userio.num_clocks)
print(scope.userio)

USERIO clocks are generated by a single PLL that is fed by one of two clocks:
1. The fixed 96 MHz USB clock.
2. The target clock, as defined by `scope.clock.clkgen_src` (i.e. can be either a Husky-generated clock or a target-generated clock).

The minimum supported input clock source frequency is 19 MHz. *This means our "standard" 7.37 MHz target clock should not be used!*

Regardless of the clock source, the supported range of generated output clock frequencies is 6.25 MHz to 800 MHz (this is a property of the FPGA).

**HOWEVER!** proper operation is not guaranteed beyond 200 MHz (Husky) / 250 MHz (Husky Plus); exceed this limit at your own risk.

Generated clocks will be phase-aligned with the USERIO clock source, and the achievable clock frequencies will depend on that source clock's frequency (e.g. how close you can get to say 12.345 MHz will depend on the source clock frequency; under the hood, the PLL multiplies and divides the source clock with integers to generate its output clocks, so not all output frequencies are possible).

Let's start using the USB clock source, and generate a 12 MHz clock on `USERIO CK`. Since 96 is a multiple of 12, we can achieve exactly 12 MHz:

In [ ]:
scope.userio.clock_source = 'usb'
scope.userio.pins[8].clock_enabled = True
scope.userio.pins[8].clock = 12e6
scope.userio.pins[8].direction = 'output'

You can use an external logic analyzer to confirm that you now have a 12 MHz clock on the `USERIO CK` pin.

It's also possible to use `scope.LA` to do this:

In [ ]:
scope.LA.enabled = True
scope.LA.clk_source = 'usb'
scope.LA.oversampling_factor = 2
scope.LA.capture_group = 'USERIO 20-pin'
scope.LA.capture_depth = 128

We have set the sampling clock to twice the USB clock (192 MHz); since the USERIO clock is one eight of the USB clock, we expect to see a clock period of `2 * 8 = 16` samples in the `scope.LA` capture.

In [ ]:
scope.LA.arm()
scope.LA.trigger_now()
userio_ck = scope.LA.extract(scope.LA.read_capture_data(), 8)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
output_notebook()

In [ ]:
xrange = list(range(scope.LA.capture_depth))
p = figure(width=900, height=150, tools='pan, box_zoom, hover, reset, save')
p.line(xrange, userio_ck,  line_color='green')
show(p)

The captured clock has a period of 16 samples, as expected.

If you have a Husky Plus, you can generate three more clocks.

Let's make these 48, 32, and 24 MHz:

In [ ]:
if scope._is_husky_plus:
    for pin, clk_div in zip(range(5,8), [2,3,4]):
        scope.userio.pins[pin].clock_enabled = True
        scope.userio.pins[pin].clock = 96e6/clk_div
        scope.userio.pins[pin].direction = 'output'
    print(scope.userio)
    scope.LA.arm()
    scope.LA.trigger_now()
    clocks = scope.LA.read_capture_data()
    userio_ck = scope.LA.extract(clocks, 8)
    userio_d7 = scope.LA.extract(clocks, 7)
    userio_d6 = scope.LA.extract(clocks, 6)
    userio_d5 = scope.LA.extract(clocks, 5)

    p = figure(width=900, height=300, tools='pan, box_zoom, hover, reset, save')
    p.line(xrange, userio_ck + 6)
    p.line(xrange, userio_d7 + 4)
    p.line(xrange, userio_d6 + 2)
    p.line(xrange, userio_d5 + 0)
    show(p)

Let's switch to using the target clock as the USERIO PLL clock source.

When you do this, you **must** also manually specify the frequency of that clock.

Remember that the minimum frequency for the PLL's source clock is 19 MHz; let's set it to 24 MHz.

In [ ]:
scope.clock.clkgen_freq = 24e6
scope.userio.clock_source_freq = 24e6
scope.userio.clock_source = 'target'

In [ ]:
scope.userio

We're still generating the same clock(s) that we previously requested.

But see what happens if we set the input clock to an "odd" number which doesn't multiply/divide so easily into our requested frequencies:

In [ ]:
scope.clock.clkgen_freq = 19.9e6
scope.userio.clock_source_freq = 19.9e6

This will give you a warning that you requested 48.000 MHz (our previously requested clock frequency, which ChipWhisperer is trying to maintain with the new clock source) but you are getting a 47.071 MHz clock frequency.

As noted earlier, the PLL cannot generate any arbitrary clock frequency. The PLL takes its input clock and feeds it through a multiplier and two dividers, all of which must be integers.

So, while a 48 MHz clock can be generated from a 96 MHz clock, it may not quite hit 48 MHz for any possible input clock. A warning is issued if the generated clock frequency is more than 1% away from the requested clock frequency.

It's important to understand that **this is only a warning**; the USERIO clock(s) are still "good"! Whether this is acceptable to you depends on **your** use case.

A third and final reminder that you shouldn't use an input clock below 19 MHz; this is what happens if you use our standard 7.37 MHz clock:

In [ ]:
scope.clock.clkgen_freq = 7.37e6
scope.userio.clock_source_freq = 7.37e6

Note that you get a warning only: we still let you do it. But be aware that you're operating the PLL outside its supported range; any generated clocks may not be "good".

If the actual input clock is very different from what you tell `scope.userio.clock_source_freq`, the PLL will likely become unlocked altogether:

In [ ]:
scope.clock.clkgen_freq = 40e6

You don't get an explicit warning in this case, but `scope.userio.clocked_locked` becomes `False`, and "PLL UNLOCKED" is added to the pin status of all affected USEIRO pins:

In [ ]:
scope.userio

With the correct frequency, things return to normal:

In [ ]:
scope.userio.clock_source_freq = 40e6
assert scope.userio.clocks_locked

# Wait, there's more!

`scope.bitbanger` can also output a clock on *any* of the USERIO pins; with Husky Plus, this clock can also go to TIO1-4.

This clock is a little bit less flexible, however, since it is not driven by its own programmable PLL.

Instead, it's derived by dividing the ADC sampling clock. The divisor is `scope.bitbanger.clk_div`.

All you need to do is assign the pin and set `scope.bitbanger.continuous_clock = True` (without the latter, the clock will only be active when you engage the bitbanger).

Here's an example:

In [ ]:
scope.bitbanger.clock_pin = 'USERIO_D0'
scope.bitbanger.continuous_clk = True
scope.bitbanger.clk_div = 4

### Diving deeper into Xilinx PLL internals

If you are generating multiple clocks, you **may** need to understand a bit more about how that is done.

*You can safely skip this section if you're happy with the range of clocks that `scope.userio` is generating for you. If you're not, be warned that what follows falls into very advanced usage. Proceed at your own risk.*

All USERIO clocks are generated by a single PLL. The generated clocks are obtained by feeding the source clock into one multiplier and two dividers. While each PLL output clock has its own "secondary" clock divider, but they **all** share a single multiplier and a "primary" divider.

This means that while it may be possible to generate a single X MHz clock and a single Y MHz clock, it may not be possible to generate both X MHz and Y  MHz at the same time.

There is also the question of how to choose the shared PLL parameters: should they be chosen in a way that gets us closer to the requested `USERIO CK` clock but further from the requested `USERIO D7` clock, or vice-versa?

There is no universal answer to that. The approach taken is that `USERIO CK` is favoured, followed by D7, D6, etc...

If you're not happy with this algorithm, you can change it! If you've never played with Xilinx PLLs before, prepare to invest a few hours into this.

1. Read and understand Xilinx's XAPP888 and UG472 documents carefully.
2. *Carefully* set the hidden multiple and divide parameters in `scope.userio._pll` directly as you wish.